# ВНИМАНИЕ! ТУТ МОГУТ БЫТЬ КРИТИЧЕСКИЕ ОШИБКИ! НЕ ИСПОЛЬЗОВАТЬ КАК ТОРГОВУЮ МОДЕЛЬ!

Создание модели , которая бы учитывала новостной фон и предыдущие цены, на основе чего формировала бы предсказание - поднимется цена или упадет

In [ ]:
from google.colab import drive
drive.mount('/content/gdrive')

Mounted at /content/gdrive


In [ ]:
!pip install -q transformers

In [ ]:
from transformers import AutoModelForSequenceClassification, AutoTokenizer
from transformers import get_scheduler
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import torch
import torch.utils.data as data
import torch.optim as optim
import torch.nn as nn
from tqdm.auto import tqdm
from sklearn.model_selection import train_test_split

device = 'cuda' if torch.cuda.is_available() else 'cpu'
device

'cpu'

In [ ]:
df = pd.read_csv('/content/gdrive/MyDrive/Colab Notebooks/course/datasets/data_coindesk_base.csv')
df = df[['SENTIMENT', 'PUBLISHED_ON']]

# df.loc[df['SENTIMENT'] == 'POSITIVE', ['SENTIMENT']] = 0
# df.loc[df['SENTIMENT'] == 'NEGATIVE', ['SENTIMENT']] = 1
# df.loc[df['SENTIMENT'] == 'NEUTRAL', ['SENTIMENT']] = 2

# df = df.sample(frac=1, random_state=52).reset_index()
df.dropna(inplace=True)

In [ ]:
df['PUBLISHED_ON_DATE'] = pd.to_datetime(df["PUBLISHED_ON"], unit='s')
df.head()

,SENTIMENT,PUBLISHED_ON,PUBLISHED_ON_DATE
0,NEUTRAL,1763120589,2025-11-14 11:43:09
1,NEUTRAL,1763120559,2025-11-14 11:42:39
2,NEGATIVE,1763119800,2025-11-14 11:30:00
3,NEGATIVE,1763119490,2025-11-14 11:24:50
4,NEGATIVE,1763119176,2025-11-14 11:19:36


In [ ]:
df_rev = df[::-1]
df_rev.head()

,SENTIMENT,PUBLISHED_ON,PUBLISHED_ON_DATE
182499,POSITIVE,1698342085,2023-10-26 17:41:25
182498,NEUTRAL,1698343101,2023-10-26 17:58:21
182497,NEUTRAL,1698343200,2023-10-26 18:00:00
182496,POSITIVE,1698343200,2023-10-26 18:00:00
182495,POSITIVE,1698343212,2023-10-26 18:00:12


In [ ]:
df_enc = pd.get_dummies(df_rev, columns=['SENTIMENT'], prefix='', prefix_sep='', dtype=int)
df_enc.head()

,PUBLISHED_ON,PUBLISHED_ON_DATE,NEGATIVE,NEUTRAL,POSITIVE
182499,1698342085,2023-10-26 17:41:25,0,0,1
182498,1698343101,2023-10-26 17:58:21,0,1,0
182497,1698343200,2023-10-26 18:00:00,0,1,0
182496,1698343200,2023-10-26 18:00:00,0,0,1
182495,1698343212,2023-10-26 18:00:12,0,0,1


In [ ]:
min_date = min(df['PUBLISHED_ON'])
max_date = max(df['PUBLISHED_ON'])
min_date, max_date

(1698342085, 1763120589)

In [ ]:
!pip install -q ccxt

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 134.1/134.1 kB 4.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 6.2/6.2 MB 52.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.6/1.6 MB 41.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 641.1/641.1 kB 25.5 MB/s eta 0:00:00


In [ ]:
import ccxt
import time
from tqdm.auto import tqdm
exchange = ccxt.kucoin({
    'enableRateLimit': True
})
timestamps = sorted(df['PUBLISHED_ON'])
timestamps[:5]

[1698342085, 1698343101, 1698343200, 1698343200, 1698343212]

In [ ]:
print(exchange.timeframes)

{'1m': '1min', '3m': '3min', '5m': '5min', '15m': '15min', '30m': '30min', '1h': '1hour', '2h': '2hour', '4h': '4hour', '6h': '6hour', '8h': '8hour', '12h': '12hour', '1d': '1day', '1w': '1week', '1M': '1month'}


In [ ]:
prices = []

symbol = 'BTC/USDT'
tf = '1m'
since = exchange.parse8601('2023-10-26T17:41:25Z')

while True:
    batch = exchange.fetch_ohlcv(symbol, tf, since=since, limit=1000)
    if not batch:
        break
    prices.extend(batch)
    since = batch[-1][0] + 60_000   # + 1 минута
    time.sleep(exchange.rateLimit / 1000)

df_prices = pd.DataFrame(prices, columns=[
    'timestamp', 'open', 'high', 'low', 'close', 'volume'
])

In [ ]:
df_prices['timestamp'] = pd.to_datetime(df_prices['timestamp'], unit='ms')
df_full = pd.merge_asof(
    df_enc.sort_values('PUBLISHED_ON_DATE'),
    df_prices.sort_values('timestamp'),
    left_on='PUBLISHED_ON_DATE',
    right_on='timestamp',
    direction='backward'
)

df_full.to_csv('/content/gdrive/MyDrive/Colab Notebooks/course/datasets/full_dataet_sentiment_prices.csv')

In [ ]:
df_full = df_full.dropna()
df_full.head()

,PUBLISHED_ON,PUBLISHED_ON_DATE,NEGATIVE,NEUTRAL,POSITIVE,timestamp,open,high,low,close,volume
1,1698343101,2023-10-26 17:58:21,0,1,0,2023-10-26 17:58:00,33955.9,33963.0,33950.9,33961.0,1.997152
2,1698343200,2023-10-26 18:00:00,0,1,0,2023-10-26 18:00:00,33944.5,33954.4,33938.0,33938.3,0.790350
3,1698343200,2023-10-26 18:00:00,0,0,1,2023-10-26 18:00:00,33944.5,33954.4,33938.0,33938.3,0.790350
4,1698343212,2023-10-26 18:00:12,0,0,1,2023-10-26 18:00:00,33944.5,33954.4,33938.0,33938.3,0.790350
5,1698343428,2023-10-26 18:03:48,0,1,0,2023-10-26 18:03:00,33968.1,33977.2,33968.1,33972.8,1.423930
